# Climbing Route Isolation

Jerry Chang owns the color-based baseline. Ford Medley owns region-based refinement and its comparison with that baseline. For each photo and target color, both sections use the same image and manually annotated ground-truth mask.

## Jerry Chang | Color-based segmentation

1. Load a wall photo as RGB and select the target hold color from a representative hold.
2. Compare channel thresholds in RGB and HSV. Account for hue wraparound when the target color straddles the ends of the hue range. Test a color-distance threshold and, where appropriate, Otsu's method on the distance image.
3. Compare erosion, dilation, opening, and closing on the binary mask. Inspect lost hold pixels, gaps, and stray wall pixels to select the baseline configuration.
4. Save the selected binary mask as a grayscale PNG with background at 0 and foreground at 255, aligned pixel for pixel with the RGB photo. Evaluate precision, recall, and IoU against manual annotations and compare behavior across lighting conditions. Ford's section uses this saved mask as its baseline.

## Ford Medley | Load the baseline and ground truth

Each case below takes a name, wall image path, Jerry's binary baseline mask path, and a manual binary ground-truth mask path. Use grayscale masks with background at zero and nonzero values for holds of the selected color. Use lossless mask files. Leave the ground-truth path as `None` to view refinements without scoring.

With the case list empty, the notebook previews the included climbing_wall.jpg. It downsizes the photo to a maximum edge of 640 pixels and builds a temporary mask by sampling a blue hold near the right side (80% across, 40% down). This lets Ford compare the refinements while Jerry's baseline and manual annotations are pending. The preview has no ground-truth scores. If the photo is absent, the notebook falls back to the synthetic wall.

The Ford cells use NumPy, SciPy, scikit-image, and Matplotlib.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy import ndimage as ndi
from skimage import color, io, segmentation, transform, util

# (name, wall image, Jerry's baseline mask, ground-truth mask or None)
cases = [
    # ("Blue route", "images/wall.jpg", "masks/blue_baseline.png", "masks/blue_truth.png"),
]


def read_mask(path, shape):
    mask = io.imread(path)
    if mask.shape != shape:
        raise ValueError(f"{path} must be a single-channel mask with shape {shape}.")
    return mask != 0


def load_case(image_path, baseline_path, truth_path):
    image = util.img_as_float(io.imread(image_path))
    if image.ndim != 3 or image.shape[2] not in (3, 4):
        raise ValueError(f"{image_path} must be an RGB or RGBA image.")
    if image.shape[2] == 4:
        image = color.rgba2rgb(image)
    shape = image.shape[:2]
    truth = read_mask(truth_path, shape) if truth_path is not None else None
    return image, read_mask(baseline_path, shape), truth


def demo_case():
    y, x = np.ogrid[:120, :160]
    first_hold = ((x - 40) / 17) ** 2 + ((y - 40) / 12) ** 2 <= 1
    second_hold = ((x - 100) / 13) ** 2 + ((y - 75) / 17) ** 2 <= 1
    truth = first_hold | second_hold
    image = np.full((120, 160, 3), [0.74, 0.70, 0.64])
    image[truth] = [0.12, 0.38, 0.81]
    image[55:68, 49:64] = [0.77, 0.20, 0.14]
    baseline = (first_hold & (x < 46)) | (second_hold & (y < 79))
    baseline[20, 115] = True
    return image, baseline, truth

def wall_preview(path):
    image = util.img_as_float(io.imread(path))
    height, width = image.shape[:2]
    scale = min(1, 640 / max(height, width))
    image = transform.resize(
        image, (round(height * scale), round(width * scale)), anti_aliasing=True
    )

    lab_image = color.rgb2lab(image)
    row = round(0.40 * (image.shape[0] - 1))
    column = round(0.80 * (image.shape[1] - 1))
    patch = lab_image[row - 2:row + 3, column - 2:column + 3]
    reference = np.median(patch, axis=(0, 1))
    distance = np.linalg.norm(lab_image - reference, axis=2)
    return image, distance <= 16, None


## Ford Medley | Region growing

Split the baseline into four-connected components and discard tiny seed components. For each remaining component, use its median CIELAB color as a fixed reference. Grow through four-connected pixels within the color-distance tolerance, so similarly colored pixels elsewhere in the photo cannot join unless they are connected to a seed. Compare tolerances for missed hold edges and spill into the wall.

In [ ]:
def region_growing(lab_image, baseline, tolerance=15, min_seed_area=5):
    connectivity = ndi.generate_binary_structure(2, 1)
    labels, count = ndi.label(baseline, structure=connectivity)
    areas = np.bincount(labels.ravel())
    refined = np.zeros_like(baseline, dtype=bool)

    for label in range(1, count + 1):
        if areas[label] < min_seed_area:
            continue
        component = labels == label
        reference = np.median(lab_image[component], axis=0)
        distance = np.linalg.norm(lab_image - reference, axis=2)
        eligible = distance <= tolerance
        refined |= ndi.binary_propagation(
            component & eligible, structure=connectivity, mask=eligible
        )

    return refined


## Ford Medley | SLIC superpixels

Partition the CIELAB image into contiguous superpixels. Keep a superpixel when enough of its pixels overlap the baseline and its average color is close to the median color of the seed pixels inside it. Vary the number of superpixels and compactness to see how well the segments follow hold boundaries, especially where holds touch or sit close together.

In [ ]:
def slic_refinement(
    lab_image, baseline, n_segments=250, compactness=10,
    min_overlap=0.20, color_tolerance=15,
):
    if not baseline.any():
        return np.zeros_like(baseline, dtype=bool)

    labels = segmentation.slic(
        lab_image, n_segments=n_segments, compactness=compactness,
        convert2lab=False, start_label=0, channel_axis=-1,
    )
    flat_labels = labels.ravel()
    areas = np.bincount(flat_labels)
    seed_counts = np.bincount(flat_labels[baseline.ravel()], minlength=len(areas))
    candidates = np.flatnonzero((areas > 0) & (seed_counts > 0)
                                & (seed_counts >= min_overlap * areas))
    keep = np.zeros(len(areas), dtype=bool)

    for label in candidates:
        region = labels == label
        mean_color = lab_image[region].mean(axis=0)
        seed_color = np.median(lab_image[region & baseline], axis=0)
        keep[label] = np.linalg.norm(mean_color - seed_color) <= color_tolerance

    return keep[labels]


## Ford Medley | Compare refinements with Jerry's baseline

Score each predicted mask against the same manual annotation using pixel-level precision, recall, and Intersection over Union (IoU). Report undefined metrics as `n/a`. The plots dim pixels outside the predicted route so missed hold edges, stray detections, and merged or adjacent holds can be inspected beside the score table. The example sweeps compare two settings per refinement method. Edit the settings in the cell below for additional comparisons.

In [ ]:
def mask_scores(prediction, truth):
    true_positive = np.count_nonzero(prediction & truth)
    denominators = (
        np.count_nonzero(prediction),
        np.count_nonzero(truth),
        np.count_nonzero(prediction | truth),
    )
    return tuple(true_positive / size if size else None for size in denominators)


def compare_methods(name, image, baseline, truth, baseline_label="Jerry: baseline"):
    lab_image = color.rgb2lab(image)
    masks = {baseline_label: baseline}
    for tolerance in (10, 20):
        masks[f"Ford: grow ({tolerance})"] = region_growing(
            lab_image, baseline, tolerance=tolerance
        )
    for count, compactness in ((150, 5), (300, 15)):
        masks[f"Ford: SLIC ({count}, {compactness})"] = slic_refinement(
            lab_image, baseline, n_segments=count, compactness=compactness
        )

    print(name)
    if truth is not None:
        print(f"{'Method':<28} {'Precision':>9} {'Recall':>9} {'IoU':>9}")
        for method, mask in masks.items():
            scores = [f"{value:.3f}" if value is not None else "n/a"
                      for value in mask_scores(mask, truth)]
            print(f"{method:<28} {scores[0]:>9} {scores[1]:>9} {scores[2]:>9}")

    panels = {"Original": image}
    if truth is not None:
        panels["Ground truth"] = image * np.where(truth[..., None], 1.0, 0.25)
    for method, mask in masks.items():
        panels[method] = image * np.where(mask[..., None], 1.0, 0.25)

    columns = 4
    rows = (len(panels) + columns - 1) // columns
    figure, axes = plt.subplots(rows, columns, figsize=(14, 3.5 * rows))
    for axis in axes.flat:
        axis.axis("off")
    for axis, (title, panel) in zip(axes.flat, panels.items()):
        axis.imshow(panel)
        axis.set_title(title)
    figure.tight_layout()
    plt.show()
    plt.close(figure)
    return masks


if cases:
    for name, image_path, baseline_path, truth_path in cases:
        compare_methods(name, *load_case(image_path, baseline_path, truth_path))
elif Path("climbing_wall.jpg").is_file():
    compare_methods(
        "Climbing wall preview - blue holds",
        *wall_preview("climbing_wall.jpg"),
        baseline_label="Temporary blue mask",
    )
else:
    compare_methods("Synthetic demo", *demo_case(), baseline_label="Synthetic baseline")
